# Python Data Analysis Practice · Delivery Operations

An online retailer receives delivery updates from several carriers. A case can appear more than once as new updates arrive. The operations lead wants to know which cases are still active, which delays need attention, and what to include in a daily briefing.

Download [delivery_exceptions.csv](data/delivery_exceptions.csv) and run the setup cell. Run Jupyter from the notebook's folder so the CSV is under `data/`. In Colab, upload the CSV through the **Files** panel and update the path as shown in the setup comment. Use Python 3.11 or newer. If packages are missing, run `%pip install numpy pandas matplotlib seaborn requests` in a notebook cell.

Write your answer in the blank cell under each question. You can reuse your work on later questions. When a question refers to **cleaned cases**, use normalized text, valid delay values, and the latest update per case. An **active** case has status `open` or `in_progress`.

Refer to [Python Tools](01_python_tools.ipynb) and the [NumPy, pandas and chart sections](02_data_and_llms.ipynb) for class examples.

## Load the CSV

| Column | What it means |
| --- | --- |
| `case_id`, `order_id` | Delivery exception and associated order. A case can have multiple updates. |
| `opened_at`, `updated_at` | When the case was opened and when this update was recorded. |
| `region`, `city`, `carrier` | Where the order was going and who was handling it. |
| `issue_type` | Reason for the case, such as `delayed`, `damaged`, or `missing_parcel`. |
| `status`, `priority` | Current state at the time of the update and its assigned priority. |
| `delay_minutes` | Reported minutes past the delivery promise. Zero is valid. Blanks, labels such as `pending sync`, and negative values are unknown for analysis. |
| `customer_message` | Short example of what the customer reported. |

The setup loads the raw export so you can explore it in the questions below.

In [ ]:
import pandas as pd

# In Colab, upload the CSV and change the path to "delivery_exceptions.csv".
raw_cases = pd.read_csv("data/delivery_exceptions.csv")
display(raw_cases.head())

## A. Explore the CSV

### 1. What came in the export?

Show the first five and last five rows, the shape, and the column names. Which column identifies an update's case? Which column tells you when that update happened?

### 2. How many cases are there?

Count the raw rows and the distinct case IDs. List the IDs that appear more than once and count how many updates each has. What do the two counts tell you about how the export was produced?

### 3. Where is the data incomplete?

Count missing values in each raw column. Then inspect `delay_minutes` and list examples of missing, invalid, and valid values. Which of those categories do blanks, `pending sync`, negative numbers, and zero belong to?

### 4. Which categories need standardization?

Show the distinct raw values of `status`, `region`, `carrier`, and `priority`. Find two labels that would be counted separately in a raw grouping even though they have the same meaning.

### 5. What happened to case 7008?

Use `.loc` to display all updates for case 7008 with just the case ID, update time, status, and delay. Sort them by update time. Then use `.iloc` to select the last row of that result. Which status should a lookup tool return?

## B. Clean and prepare the data

### 6. Can you normalize the text labels?

Create a copy and trim spaces and lowercase the region, carrier, issue type, status, and priority columns. Use a mapping to turn both `in progress` and `in-progress` into `in_progress`. Compare the distinct status labels before and after cleaning.

### 7. Can you turn a messy delay column into numbers?

Convert `delay_minutes` to numbers with pandas. Treat blanks, unparseable text, and negative values as missing. Keep zero as a valid measurement. How many measurements became missing? Inspect cases 7004, 7005, and 7006 to show the difference.

### 8. How much time passed before each update?

Parse `opened_at` and `updated_at` as datetimes. Create a column giving the minutes between opening and that update. Find the three updates with the largest elapsed time and explain what this measure tells you about a case.

### 9. Which update should represent each case?

Sort by `updated_at` and keep the latest row for each `case_id`. The export is shuffled, so use the timestamp to decide which row to keep. Compare the status counts before and after deduplication.

### 10. Can you make the cleaning reusable?

Package the cleaning and deduplication into `prepare_cases(raw_cases: pd.DataFrame) -> pd.DataFrame`. Return a new table and leave the input intact. Print a short summary of how many rows came in, how many unique cases remained, and how many of those cases had unknown delays.

## C. Practise Python functions for agent tools

### 11. Can you validate a case ID from a request?

Write `parse_case_id(value: object) -> int`. Accept positive integers and digit strings after trimming spaces. Raise `ValueError` for other values, including booleans. Try it with `7001`, `" 7001 "`, `"ORD-8401"`, and `True`.

### 12. Can you look up the latest status?

Convert the cleaned table to a list of dictionaries. Write a lookup function that accepts a case ID and returns a dictionary with the case ID, status, carrier, and issue type. Return an error dictionary if the ID is missing from the dataset, and try a known ID and an absent one.

### 13. Which cases should the operations team review first?

Write a function that accepts the records, a region, and a delay threshold. Return active cases in that region with a known delay at or above the threshold. Use a lambda to order them by longest delay first, with smaller case IDs first when delays tie. Compare the North region at 120 and 240 minutes.

### 14. How can one run avoid reviewing a case twice?

Build a `DeliveryRun` class with a set on each instance. Give it methods to mark an ID as seen, check whether it was seen, and count distinct IDs. Create two instances, mark an ID twice in the first, and show what each instance remembers.

### 15. Can you turn case records into short descriptions?

Write a named function that accepts one case and returns a line such as `Case 7008 | north | resolved`. Apply it to a list of records using Python's built-in `map()`. Use `functools.reduce()` to total the known delays for active cases and compare the result with `sum()`.

## D. Analyze the backlog with NumPy and pandas

### 16. How many delays meet each threshold?

Build a NumPy array of known delays from active cases. Use boolean masks to count the delays at or above 60, 120, and 240 minutes. Also report the minimum, median, and maximum. How many active cases were excluded because their delays were unknown?

### 17. What happens when you combine two conditions?

Use `.loc` to find high or urgent active cases with a known delay of at least 120 minutes. Show only the case ID, region, carrier, priority, and delay. Sort the result and use `.iloc` to display the first three matching rows.

### 18. Which carriers have the most active cases?

Group active cases by carrier. For each carrier, show the number of active cases, the number with a known delay, and the average known delay. Why might `.size()` and `.count()` give different numbers?

### 19. Can you assign a label to each known delay?

Use `Series.map()` with a dictionary to give each priority a readable label. Then use `Series.apply()` with a named function to label delays as `unknown`, `under 120 minutes`, or `120 minutes or more`. Show the counts for each delay label among active cases.

### 20. Which region and carrier combinations need attention?

Build a grouped table by region and carrier for active cases. Show the active count and the number at or above 120 minutes. Calculate the percentage meeting the threshold **among known delays** and use a missing result when a group has no known delays. Which combinations look different when you compare counts with percentages?

## E. Choose and explain a chart

### 21. Which carrier has the most active cases?

Make a Seaborn count plot using active cases by carrier. Add labels above the bars and compare them with your grouped counts. Give the chart a title and label both axes.

### 22. How are known delays distributed?

Plot a histogram of known delays for active cases. Draw a line at 120 minutes, then compare the chart with the exact count from a mask. How could changing the histogram bins change what you notice?

### 23. Which regions have a wider spread of delays?

Make a Seaborn box plot of known active delays by region. Compare medians and spread. Choose one region with a point beyond a whisker and look up the case or cases that may explain it.

### 24. How did new exceptions change over time?

Count unique cases by the calendar day on which they opened. Plot the counts in date order as a line chart. Identify the busiest day. Explain why cases opened on a day and cases still active on that day are different measures.

### 25. Do longer recorded delays come with longer update times?

On the cleaned table, make a scatter plot of delay minutes against minutes from opening to the latest update. Use region as the color and exclude unknown delays. Describe one pattern or exception, and name another factor you would want to check before drawing a conclusion.

## F. Build Python tools and reports

### 26. Can your program exchange a tool request as JSON?

Start with `{"tool": "get_case_status", "case_id": "7008"}` as JSON text. Parse it, call the lookup tool, and convert the returned dictionary back to JSON text. Decide what error the application should return for malformed JSON, an unsupported tool, and an absent case. Demonstrate one request for each outcome.

### 27. Can you wrap an HTTP request in a Python function?

Use the class practice endpoint at `https://jsonplaceholder.typicode.com/todos/1`. Write a function that accepts a task ID, sends a GET request with a timeout, checks the response status, and returns a dictionary with the ID, title, and completion state. Decide how to return a useful error if the request cannot complete. You can use a sample response dictionary when practising offline.

### 28. Can you save and reload a daily report?

Build a dictionary with the total active cases, unknown delays, counts by carrier, and number of active cases delayed at least 120 minutes. Save it as `practice_delivery_report.json` using UTF-8. Reload it and show that the values match. Convert any pandas or NumPy values to JSON-compatible Python types where needed.

### 29. Can your tool validate a delay search request?

Start with `{"tool": "find_delayed_cases", "region": " North ", "min_delay": 120}` as JSON text. Normalize the region and check that it exists in the cleaned table. Accept a non-negative whole number for the threshold. Then reuse your delay search function and return the matching case IDs and count in a dictionary. Return a clear error for an unknown region or invalid threshold.

### 30. Can you build a daily delivery report from a CSV?

Write a function that takes a CSV path and a delay threshold. Load and clean the data, then return a dictionary with the number of active cases, how many have unknown delays, counts by carrier, and how many known delays meet the threshold. Use an f-string to write a one-sentence briefing from these values. Handle a missing file with a clear message. Run the report for 120 and 240 minutes and explain what changed.